In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('local[*]')
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 20:23:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

dept_data = [
    ["101", "Sales", "NYC", "US", "1000000"],
    ["102", "Marketing", "LA", "US", "900000"],
    ["103", "Finance", "London", "UK", "1200000"],
    ["104", "Engineering", "Beijing", "China", "1500000"],
    ["105", "Human Resources", "Tokyo", "Japan", "800000"],
    ["106", "Research and Development", "Perth", "Australia", "1100000"],
    ["107", "Customer Service", "Sydney", "Australia", "950000"]
]

dept_schema = "department_id string, department_name string, city string, country string, budget string"

In [3]:
emp_df = spark.createDataFrame(data=emp_data, schema=emp_schema)
dept_df = spark.createDataFrame(data=dept_data, schema=dept_schema)

emp_df.printSchema()
dept_df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)

root
 |-- department_id: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- budget: string (nullable = true)



In [4]:
emp_df.rdd.getNumPartitions()

15

In [5]:
dept_df.rdd.getNumPartitions()

15

In [6]:
from pyspark.sql.functions import spark_partition_id

emp_df.withColumn('partition_id', spark_partition_id()).show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|           0|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|           1|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|           2|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|           2|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|           3|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|           4|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|           5|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|           5|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|      

In [7]:
emp_df.select('department_id').distinct().count()

7

In [8]:
emp_df = emp_df.repartition(4, 'department_id')

emp_df.rdd.getNumPartitions()

4

In [9]:
emp_df.withColumn('partition_id', spark_partition_id()).orderBy('department_id').show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|           3|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|           3|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|           3|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|           0|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|           0|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|           0|
|        020|          102|    Grace Kim| 32|Female| 53000|2018-11-01|           0|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|           3|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|      

In [13]:
from pyspark.sql.functions import col

emp_joined_df = emp_df.alias('e').join(dept_df.alias('d'), how='inner', on=col('e.department_id')==col('d.department_id')).select('e.employee_id', 'e.department_id', 'e.name', 'e.age', 'e.gender', 'e.salary', 'e.hire_date', 'd.department_name', 'd.city', 'd.country', 'd.budget')

In [14]:
emp_joined_df.show()

+-----------+-------------+-------------+---+------+------+----------+--------------------+-------+---------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|     department_name|   city|  country| budget|
+-----------+-------------+-------------+---+------+------+----------+--------------------+-------+---------+-------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|               Sales|    NYC|       US|1000000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|               Sales|    NYC|       US|1000000|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|               Sales|    NYC|       US|1000000|
|        014|          107|    Emily Lee| 26|Female| 46000|2019-01-01|    Customer Service| Sydney|Australia| 950000|
|        016|          107|  Kelly Zhang| 30|Female| 49000|2018-04-01|    Customer Service| Sydney|Australia| 950000|
|        010|          104|     Lisa Lee| 27|Female| 470